# Agent training

In [ ]:
from circle_environment import CircleEnv
from stable_baselines3 import PPO, A2C, DQN
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.evaluation import evaluate_policy
from datetime import datetime
import os
import logging
        
def configure_logging(log_file_path, console_log_level=logging.INFO):
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
    handlers = []
    if console_log_level is not None:
        # define a Handler which writes INFO messages or higher to the sys.stderr
        console_handler = logging.StreamHandler()
        console_handler.setLevel(console_log_level)
        console_handler.setFormatter(formatter)
        handlers.append(console_handler)
    if log_file_path is not None:
        # create file handler which logs even debug messages
        file_handler = logging.FileHandler(log_file_path, mode="a")
        file_handler.setLevel(logging.DEBUG)
        file_handler.setFormatter(formatter)
        handlers.append(file_handler)
    # add the handlers to the (root) logger
    logging.basicConfig(level=logging.DEBUG, 
                    handlers=handlers,
                    force=True) # force=True overwrites the logging configuration so that we can change the logfile name

def get_log_level(training_or_evaluation):
    match training_or_evaluation:
        case "training":
            return None
        case "evaluation":
            return logging.INFO
        case _:
            return None

def setup_logging(algorithm, version, map, n_vehicles, training_or_evaluation, model_load_path=None):
    # Create a unique identifier for this training run
    current_time = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    log_id = f"{version}_{map}_{algorithm}_{n_vehicles}vehicles_{training_or_evaluation}_{current_time}"    
    new_model_id = f"{version}_{map}_{algorithm}_{current_time}"
    # if a model path is given, i.e. an existing model is evaluated or trained further
    if model_load_path is not None:
        current_model_id = model_load_path.split("/")[-1].rsplit(".", 1)[0] # Extract id from model_load_path
    else:
        current_model_id = new_model_id
    
    # Set up non existing directories
    model_dir = f"./models/{current_model_id}"
    log_dir = f"{model_dir}/{training_or_evaluation}"
    os.makedirs(model_dir, exist_ok=True)
    os.makedirs(log_dir, exist_ok=True)
    
    model_save_path = f"{model_dir}/{new_model_id}.zip"
    py_log_path = f"{log_dir}/{log_id}.log"
    console_log_level = get_log_level(training_or_evaluation)
    configure_logging(log_file_path=py_log_path, console_log_level=console_log_level)
    return model_dir, model_save_path

def train_model(algorithm, policy, version, map, n_vehicles, n_steps):
    # initiate environment
    env = CircleEnv(render_mode=None, vehicles_to_spawn=n_vehicles)
    # setup logging
    model_dir, model_save_path = setup_logging(algorithm, version, map, n_vehicles, "training")
    # Train the agent
    match algorithm:
        case "PPO":
            model = PPO(policy, env, verbose=1, tensorboard_log=model_dir)
        case "A2C":
            model = A2C(policy, env, verbose=1, tensorboard_log=model_dir)
        case "DQN":
            model = DQN(policy, env, verbose=1, tensorboard_log=model_dir)
        case _:
            raise ValueError("Invalid model type")
    tb_log_name = "tensorboard"
    model.learn(n_steps, tb_log_name=tb_log_name, callback=StepLoggerCallback())
    model.save(model_save_path)
    env.close()
    
def load_model(model_path, algorithm, env):
    match algorithm:
        case "PPO":
            model = PPO.load(model_path, env=env)
        case "A2C":
            model = A2C.load(model_path, env=env)
        case "DQN":
            model = DQN.load(model_path, env=env)
        case _:
            raise ValueError("Invalid model type")
    return model

def further_train_model(model_load_path, algorithm, version, map, n_vehicles, n_steps):
    # initiate environment
    env = CircleEnv(render_mode=None, vehicles_to_spawn=n_vehicles)
    # setup logging
    model_dir, model_save_path = setup_logging(algorithm, version, map, n_vehicles, "training", model_load_path)
    # Train the agent
    model = load_model(model_load_path, algorithm, env)
    tb_log_name = "tensorboard"
    model.learn(n_steps, tb_log_name=tb_log_name, callback=StepLoggerCallback())
    model.save(model_save_path)
    env.close()

def evaluate_model(model_load_path, algorithm, version, map, n_vehicles, n_episodes):
    # initiate environment
    env = CircleEnv(render_mode="human", vehicles_to_spawn=n_vehicles)
    # setup logging
    setup_logging(algorithm, version, map, n_vehicles, "evaluation", model_load_path)
    # Load saved model
    model = load_model(model_load_path, algorithm, env)
    # Evaluate the agent
    mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=5)
    print(f"mean_reward: {mean_reward}, std_reward: {std_reward}")
    env.close()

class StepLoggerCallback(BaseCallback):
    """ used to log the current agent step number """
    def __init__(self, verbose=0):
        super(StepLoggerCallback, self).__init__(verbose)
    
    def _on_step(self) -> bool:
        # TODO: num_timesteps an env weitergeben, damit der env-logger auch die sb3-timesteps loggen kann. Dann braucht der sb3-Logger das auch nicht mehr mitloggen.
        # self.logger.record("current_step", self.num_timesteps)
        # self.logger.dump(self.num_timesteps)
        logger = logging.getLogger("rl")
        logger.debug(f"Agent Step {self.num_timesteps}")
        return True

In [ ]:
# Train new model
train_model("A2C", "MultiInputPolicy", "v0.3", "circle", n_vehicles=5, n_steps=20000)

In [ ]:
# Train saved model further
model_path = "models/v0.3_circle_a2c_2024-09-18_23-58-11/v0.3_circle_a2c_2024-09-19_08-37-20.zip"
further_train_model(model_path, "A2C", "v0.3", "circle", n_vehicles=5, n_steps=20000)

In [ ]:
# Quick evaluation
model_path = "models/v0.3_circle_a2c_2024-09-18_23-58-11/v0.3_circle_a2c_2024-09-18_23-58-11.zip"
evaluate_model(model_path, "A2C", "v0.3", "circle", n_vehicles=5, n_episodes=5)

In [ ]:
from stable_baselines3 import PPO, A2C, DQN
from circle_environment import CircleEnv

# Observe execution of trained agent in GUI
env = CircleEnv(render_mode="human", vehicles_to_spawn=5)
model = A2C.load(model_save_name, env)

num_steps = 5
observation, info = env.reset()
for t in range(num_steps):
        actions, _ = model.predict(observation, state=None, deterministic=False)
        observation, reward, terminated, truncated, info = env.step(actions)

env.close()

In [ ]:
# Random actions to compare with the agent
env = CircleEnv(render_mode="human", log_level="info", vehicles_to_spawn=5)
observation, info = env.reset()
for _ in range(5):
    action = env.action_space.sample() # select a random action
    observation, reward, terminated, truncated, info = env.step(action)
    # if terminated or truncated:
        # observation, info = env.reset()
        
env.close()

---
# Test charging stop removal

In [ ]:
from circletest import Simulation

cs_id = "cs_0"
vehicle_id = "myVehicle0"
simulation = Simulation(gui=False)
simulation.add_vehicles()

def print_stops():
    stops = simulation.get_stops(vehicle_id)
    print(f"Stops: {stops}")

# starten
simulation.step()
print_stops()
# rerouten
print("REROUTE")
simulation.reroute_for_charging(vehicle_id, cs_id)
print_stops()
# stop removen
print("REMOVE STOP")
simulation.remove_charging_stop(vehicle_id)
print_stops()

simulation.close()